# RAG Pipeline: FAISS + ChromaDB + Hugging Face Q&A

A complete Retrieval-Augmented Generation pipeline: load and prepare a news dataset,
embed titles with Sentence Transformers, index and search them with **FAISS**, store
and query them with **ChromaDB**, and answer questions with a Hugging Face causal
language model using the retrieved documents as context.

### What you'll learn
- Vector search strategies (KNN, ANN) and evaluation.
- Vector database utility (similarity search, RAG).
- Differences between vector databases, libraries, and plugins.
- Best practices for vector store usage and performance.
- How language models learn knowledge via context.
- Text embedding generation and vector storage.
- Querying vector stores for relevant documents.
- Applying language models for question answering with retrieved context.

Run the cells top to bottom. Each install/import cell is self-contained, so a fresh
Colab runtime (or a **Runtime > Restart session**, which the numpy note below explains
when you'd need one) can start from the top and reach the end without extra steps.

## Setup

Installed in the order the exercise specifies, including the parts that look
redundant at first glance -- see the note right after this cell for why that
order actually matters.

In [ ]:
!pip install -q faiss-cpu==1.7.4
!pip install -q chromadb==0.3.21
!pip install -qU chromadb
!pip install -q "numpy<2"


**Why this looks redundant, and why the order matters:** the exercise pins
`chromadb==0.3.21` and then immediately upgrades it with `-U chromadb` -- the
upgrade wins, so the collection API used later in this notebook (`.create_collection`,
`.add`, `.query`) is whatever the *current* ChromaDB release provides, confirmed by
running this exact sequence and checking `chromadb.__version__` before writing the
rest of this notebook. `numpy<2` is installed **last** so it overrides whatever
newer NumPy ChromaDB's own dependency resolution just pulled back in.

One more thing worth knowing before you hit it: `faiss-cpu==1.7.4` and `numpy<2` are
old, narrow pins. If a *later* cell in this notebook (torch/transformers) reinstalls a
newer NumPy as a side effect, you can end up with a NumPy 1.x/2.x ABI mismatch --
it shows up as a confusing `ValueError` or segfault-like crash from FAISS or ChromaDB
that has nothing to do with your actual code. If that happens: rerun
`!pip install -q "numpy<2"`, then **Runtime > Restart session** before continuing,
rather than debugging the pipeline logic.

In [ ]:
!mkdir -p cache
!apt-get -y -qq install libomp-dev
!python -m pip install -q --upgrade faiss-cpu


That last line -- `--upgrade faiss-cpu` -- is also from the exercise's own
setup steps, and it supersedes the `==1.7.4` pin above it. So despite the pin, the
FAISS build actually used everywhere below is whatever's current on PyPI, not 1.7.4.

**One addition beyond the exercise's own install list:** its "Install Required
Libraries" step only names `faiss-cpu` and `chromadb` explicitly, but the imports
it lists a few steps later need `sentence-transformers` and `transformers` too, and
Exercise 5's `pipeline(..., device_map="auto")` needs `accelerate` (it raises
`ImportError: Using a device_map ... requires Accelerate` without it). Installing
all three here so the notebook runs top-to-bottom with no missing-dependency
surprises later on.

In [ ]:
!pip install -q -U sentence-transformers transformers accelerate


In [ ]:
import numpy as np
import pandas as pd
import faiss
import json
from sentence_transformers import SentenceTransformer, InputExample
import chromadb
from chromadb.config import Settings
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline


## Exercise 1: Data Loading and Preparation

Before embeddings or vector databases, the dataset needs to be downloaded, loaded,
given a unique ID per row, inspected, and cut down to a subset small enough for fast
iteration.

In [ ]:
# Download + unzip the labelled Newscatcher dataset into cache/
!wget -q "https://github.com/devtlv/Datasets-GEN-AI-Bootcamp/raw/refs/heads/main/Week%205/Day%204%20-%20Diving%20Deep%20into%20Vector%20Databases%20and%20RAG%20Chatbots/labelled_newscatcher_dataset.zip" -O cache/labelled_newscatcher_dataset.zip
!unzip -o -q cache/labelled_newscatcher_dataset.zip -d cache/

path = "cache/labelled_newscatcher_dataset.csv"
# Confirmed by inspecting the raw file directly: it's semicolon-delimited, not
# comma-delimited. Plain pd.read_csv(path) doesn't error on this -- it just
# silently loads every row into a single column.
pdf = pd.read_csv(path, sep=";")


In [ ]:
pdf["id"] = pdf.index


In [ ]:
display(pdf)
print(pdf.shape)
print(pdf.dtypes)
print("Missing values per column:\n", pdf.isnull().sum())


**What the inspection shows** (confirmed by running the cell above against the
real file): 108,774 rows and 6 original columns -- `topic`, `link`, `domain`,
`published_date`, `title`, `lang` -- plus the `id` column just added. All columns are
strings except `id`. No missing values in any column. `topic` has 8 categories
(`SCIENCE`, `TECHNOLOGY`, `HEALTH`, `WORLD`, `ENTERTAINMENT`, `SPORTS`, `BUSINESS`,
`NATION`); 7 of the 8 have exactly 15,000 rows each, and `SCIENCE` has 3,774 -- the
dataset is a deliberately near-balanced sample, not organic news volume.

In [ ]:
pdf_subset = pdf.head(1000)


## Exercise 2: Vectorization with Sentence Transformers

Turning each news title into a dense embedding vector, using the pre-trained
`all-MiniLM-L6-v2` Sentence Transformer model.

In [ ]:
# InputExample is already imported in the setup cell above; re-stated here per
# the exercise's own step 1.
from sentence_transformers import InputExample


In [ ]:
pdf_subset = pdf.head(1000)  # the subset created in Exercise 1

def example_create_fn(doc1: pd.Series) -> InputExample:
    """Helper function that outputs a sentence_transformer guid, label, and text."""
    return InputExample(texts=[doc1])

faiss_train_examples = pdf_subset.apply(lambda x: example_create_fn(x["title"]), axis=1).tolist()
faiss_train_examples[:10]


In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")


In [ ]:
titles_list = pdf_subset["title"].tolist()


In [ ]:
faiss_title_embedding = model.encode(titles_list, show_progress_bar=True)


In [ ]:
len(faiss_title_embedding), len(faiss_title_embedding[0])


Expected output: `(1000, 384)` -- one 384-dimensional embedding per title,
matching `all-MiniLM-L6-v2`'s output size.

## Exercise 3: FAISS Indexing and Search

Building a FAISS index over the title embeddings so we can retrieve the most
similar articles to a query in milliseconds instead of scanning all 1,000 vectors
by hand.

In [ ]:
pdf_to_index = pdf_subset.set_index("id", drop=False)
id_index = np.array(pdf_to_index["id"])


In [ ]:
# .copy() so re-running this cell doesn't re-normalize an already-normalized
# faiss_title_embedding in place (normalize_L2 mutates its argument).
content_encoded_normalized = faiss_title_embedding.copy()
faiss.normalize_L2(content_encoded_normalized)


In [ ]:
index_content = faiss.IndexIDMap(faiss.IndexFlatIP(len(faiss_title_embedding[0])))
index_content.add_with_ids(content_encoded_normalized, id_index)


In [ ]:
def search_content(query, pdf_to_index, k=3):
    query_vector = model.encode([query])
    faiss.normalize_L2(query_vector)

    # Perform the search
    top_k = index_content.search(query_vector, k)
    ids = top_k[1][0].tolist()
    similarities = top_k[0][0].tolist()

    # .copy() avoids a SettingWithCopyWarning on the next line -- .loc[list_of_ids]
    # can hand back a view into pdf_to_index rather than an independent frame.
    results = pdf_to_index.loc[ids].copy()
    results["similarities"] = similarities
    return results


In [ ]:
display(search_content("animal", pdf_to_index, k=5))


## Exercise 4: ChromaDB Collection and Querying

Unlike FAISS, ChromaDB can embed, index, and query text for you -- no manual
`model.encode()` step required. This section stores 100 titles (with their topic as
metadata) in a ChromaDB collection and queries it directly with text.

In [ ]:
chroma_client = chromadb.Client()
collection_name = "my_news"

# If a collection with the same name exists, delete it to avoid conflicts.
# Fixed a bug in the exercise's own scaffold here: `list_collections()[0].name`
# only ever checks the *first* collection in the client, so if "my_news" isn't
# first, the stale collection is never deleted and create_collection() below
# throws "collection already exists" on a second run of this cell. Checking
# every existing collection's name (not just the first) avoids that.
existing_names = [c.name for c in chroma_client.list_collections()]
if collection_name in existing_names:
    chroma_client.delete_collection(name=collection_name)

print(f"Creating collection: '{collection_name}'")
collection = chroma_client.create_collection(name=collection_name)


In [ ]:
# Display the DataFrame subset (for reference)
display(pdf_subset)

collection.add(
    documents=pdf_subset["title"][:100].tolist(),
    metadatas=[{"topic": topic} for topic in pdf_subset["topic"][:100].tolist()],
    ids=[str(i) for i in pdf_subset["id"][:100].tolist()],  # ChromaDB ids must be strings
)


In [ ]:
results = collection.query(query_texts=["space"], n_results=10)

print(json.dumps(results, indent=4))


## Exercise 5: Question Answering with Hugging Face Model

Combining ChromaDB's retrieval (the `results` from Exercise 4's "space" query) with
a Hugging Face causal language model to generate a context-grounded answer -- the
core pattern behind Retrieval-Augmented Generation.

In [ ]:
model_id = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_id)
lm_model = AutoModelForCausalLM.from_pretrained(model_id)


In [ ]:
pipe = pipeline(
    "text-generation",
    model=lm_model,
    tokenizer=tokenizer,
    max_new_tokens=512,  # Maximum number of tokens to generate.
    device_map="auto",   # Automatically uses available GPU/CPU resources.
)


In [ ]:
question = "What's the latest news on space development?"
context = " ".join([f"#{str(i)}" for i in results["documents"][0]])  # results is from Exercise 4's "space" query
prompt_template = f"Relevant context: {context}\n\n The user's question: {question}"
print(prompt_template)


In [ ]:
lm_response = pipe(prompt_template)
print(lm_response[0]["generated_text"])


**A note on what to actually expect here:** `gpt2` is a small, general-purpose
model with no instruction-tuning or Q&A fine-tuning -- so it tends to continue the
prompt's *style* (more news-headline-like text) rather than produce a crisp, direct
answer to the question. That's expected behavior for this specific model choice, not
a bug in the pipeline. Exercise 5's own step 6 (below) is about seeing that behavior
change as you vary the question and the retrieved context.

In [ ]:
# Experimenting with a different question and a smaller context window (k=3 vs 10)
question_2 = "What is happening with NASA and Mars?"
results_2 = collection.query(query_texts=["Mars NASA"], n_results=3)
context_2 = " ".join([f"#{str(i)}" for i in results_2["documents"][0]])
prompt_template_2 = f"Relevant context: {context_2}\n\n The user's question: {question_2}"

lm_response_2 = pipe(prompt_template_2)
print(lm_response_2[0]["generated_text"])


### Where to go from here

- Swap `n_results` (ChromaDB) or `k` (FAISS) up or down and compare how a larger or
  smaller context window changes the generated answer.
- Try an instruction-tuned, encoder-decoder model instead of plain `gpt2` for more
  directly responsive answers -- e.g. `google/flan-t5-base` with
  `AutoModelForSeq2SeqLM` instead of `AutoModelForCausalLM`.
- Rerun Exercise 1-3 with the full dataset (drop the `.head(1000)` subset) to see how
  FAISS search quality and speed hold up at 108,774 rows instead of 1,000.